# 6. Memory PII 보호

이 노트북은 [`2. langgraph/2. Memory/1. MemorySaver.ipynb`](../2.%20langgraph/2.%20Memory/1.%20MemorySaver.ipynb)의 **Memory가 적용된 챗봇(`chat_node_with_memory` + `MemorySaver`) 코드를 그대로 재사용**해서, 대화 내용을 저장하기 전에 걸러내지 않으면 어떤 문제가 생기는지 확인합니다.

- **A 코드**: 강의의 Memory 챗봇을 그대로 사용 → 개인정보가 담긴 대화가 저장소에 그대로 남는 문제 확인
- **B 코드**: 같은 그래프/체크포인터 구조를 그대로 두고, 메시지를 저장하기 **직전에** 개인정보만 마스킹 → 저장소에 개인정보가 남지 않는 것을 확인

## 왜 Memory에 개인정보가 위험한가요?

`MemorySaver`(또는 SQLite/PostgreSQL 기반 체크포인터)는 사용자가 보낸 메시지를 **있는 그대로** 저장합니다. 그런데 사용자는 대화 중에 전화번호, 주민등록번호, 카드번호 같은 개인정보(PII)를 아무렇지 않게 입력하는 경우가 많습니다("내 번호는 010-1234-5678이야, 여기로 연락해줘" 같은 자연스러운 문장 안에요).

이게 왜 문제가 될까요?

- 체크포인터가 파일/DB에 저장되는 순간, 그 저장소에 접근할 수 있는 사람(운영자, 백업 담당자, 혹은 저장소가 뚫렸을 때의 공격자)은 대화 상대가 입력한 개인정보를 **모델이 뭐라고 답했는지와 무관하게** 그대로 열람할 수 있습니다.
- LLM의 답변만 검사하는 출력 Guardrail(2번 노트북)은 **저장되는 시점의 원본 메시지**까지는 지켜주지 못합니다. Memory는 "모델이 생성한 결과물"이 아니라 "사용자가 입력한 원본"을 저장하기 때문입니다.

그래서 이번엔 저장 **직전**에 개인정보를 걸러내는 별도의 방어선이 필요합니다.

## 실습 준비: Memory가 적용된 챗봇

> 아래 셀은 [`1. MemorySaver.ipynb`](../2.%20langgraph/2.%20Memory/1.%20MemorySaver.ipynb)의 "2. Memory가 적용된 LangGraph 예시"를 그대로 옮긴 것입니다.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    model_kwargs={
        "top_p": 1.0,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
    },
    max_tokens=2000,
)

In [3]:
from langgraph.graph import MessagesState

def chat_node_with_memory(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [4]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

graph_with_memory = StateGraph(MessagesState)
graph_with_memory.add_node("chat", chat_node_with_memory)
graph_with_memory.add_edge(START, "chat")
graph_with_memory.add_edge("chat", END)

memory = MemorySaver()
app_with_memory = graph_with_memory.compile(checkpointer=memory)

## [A] 가드레일 없는 버전

사용자가 자기소개를 하면서 개인정보를 자연스럽게 흘립니다. `invoke()`에 넘기는 메시지는 **사용자가 입력한 그대로**입니다.

In [5]:
import uuid
from langchain_core.messages import HumanMessage

memory_id_a = str(uuid.uuid4())
config_a = {"configurable": {"thread_id": memory_id_a}}

user_input = "안녕하세요, 저는 김철수입니다. 제 전화번호는 010-1234-5678이고 주민등록번호는 901231-1234567이에요. 이 번호로 연락 부탁드려요."

result = app_with_memory.invoke(
    {"messages": [HumanMessage(content=user_input)]},
    config=config_a,
)
print("AI:", result["messages"][-1].content)

AI: 죄송하지만, 저는 개인 정보를 저장하거나 이용할 수 없습니다. 제공해 주신 전화번호와 주민등록번호는 즉시 삭제하도록 하겠습니다. 다른 도움이 필요하시면 알려 주세요.


### [A] 실행: 저장소를 직접 들여다보면?

사용자에게 뭐라고 답했는지와 별개로, **체크포인터에 실제로 저장된 원본 메시지**를 확인합니다.

In [6]:
stored_state = app_with_memory.get_state(config_a)
stored_messages = stored_state.values["messages"]

print("=== 체크포인터에 저장된 원본 메시지 ===")
for m in stored_messages:
    print(f"[{m.type}] {m.content}")

=== 체크포인터에 저장된 원본 메시지 ===
[human] 안녕하세요, 저는 김철수입니다. 제 전화번호는 010-1234-5678이고 주민등록번호는 901231-1234567이에요. 이 번호로 연락 부탁드려요.
[ai] 죄송하지만, 저는 개인 정보를 저장하거나 이용할 수 없습니다. 제공해 주신 전화번호와 주민등록번호는 즉시 삭제하도록 하겠습니다. 다른 도움이 필요하시면 알려 주세요.


**문제 확인**: 저장된 첫 번째 메시지(`[human]`)에 전화번호와 주민등록번호가 **원문 그대로** 들어 있을 것입니다. 이 정보는 이제 이 애플리케이션의 체크포인터(메모리/파일/DB)에 영구히 남아, 이 저장소에 접근하는 누구나 열람할 수 있습니다. `llm`이 답변에서 이 정보를 언급했는지 여부와는 완전히 별개의 문제입니다.

## [B] 가드레일 추가: 저장 전 PII 마스킹

**그래프 구조(`chat_node_with_memory`, `StateGraph`, `MemorySaver`)는 한 글자도 바꾸지 않습니다.** 대신 `invoke()`에 메시지를 넘기기 **직전**에 PII를 마스킹하는 함수만 하나 추가합니다.

In [7]:
import re

PII_PATTERNS = [
    (r"\d{6}-\d{7}", "[주민등록번호 마스킹됨]"),
    (r"01[016789]-\d{3,4}-\d{4}", "[전화번호 마스킹됨]"),
    (r"\d{4}-\d{4}-\d{4}-\d{4}", "[카드번호 마스킹됨]"),
]


def mask_pii(text: str) -> str:
    """메시지가 Memory에 저장되기 전에 개인정보 패턴을 마스킹합니다."""
    masked = text
    for pattern, replacement in PII_PATTERNS:
        masked = re.sub(pattern, replacement, masked)
    return masked

In [8]:
memory_id_b = str(uuid.uuid4())
config_b = {"configurable": {"thread_id": memory_id_b}}

user_input_b = "안녕하세요, 저는 김철수입니다. 제 전화번호는 010-1234-5678이고 주민등록번호는 901231-1234567이에요. 이 번호로 연락 부탁드려요."

# 핵심: invoke에 넘기기 전에 마스킹 -> 마스킹된 버전이 그래프에 들어가고, 그대로 체크포인터에 저장됨
safe_input_b = mask_pii(user_input_b)
print("실제 저장/전송되는 메시지:", safe_input_b)

result_b = app_with_memory.invoke(
    {"messages": [HumanMessage(content=safe_input_b)]},
    config=config_b,
)
print("AI:", result_b["messages"][-1].content)

실제 저장/전송되는 메시지: 안녕하세요, 저는 김철수입니다. 제 전화번호는 [전화번호 마스킹됨]이고 주민등록번호는 [주민등록번호 마스킹됨]이에요. 이 번호로 연락 부탁드려요.
AI: 죄송하지만, 저는 개인 정보를 수집하거나 저장하지 않으며, 전화나 메시지 등으로 직접 연락을 드릴 수 없습니다.  

개인 정보(전화번호, 주민등록번호 등)는 공개적인 채팅 환경에 공유하지 않는 것이 안전합니다. 필요하신 경우 해당 정보를 안전한 방법으로 직접 전달해 주시기 바랍니다.  

다른 도움이 필요하시면 언제든 알려 주세요!


### [B] 실행: 저장소를 다시 확인

In [9]:
stored_state_b = app_with_memory.get_state(config_b)
stored_messages_b = stored_state_b.values["messages"]

print("=== 체크포인터에 저장된 메시지 ===")
for m in stored_messages_b:
    print(f"[{m.type}] {m.content}")

=== 체크포인터에 저장된 메시지 ===
[human] 안녕하세요, 저는 김철수입니다. 제 전화번호는 [전화번호 마스킹됨]이고 주민등록번호는 [주민등록번호 마스킹됨]이에요. 이 번호로 연락 부탁드려요.
[ai] 죄송하지만, 저는 개인 정보를 수집하거나 저장하지 않으며, 전화나 메시지 등으로 직접 연락을 드릴 수 없습니다.  

개인 정보(전화번호, 주민등록번호 등)는 공개적인 채팅 환경에 공유하지 않는 것이 안전합니다. 필요하신 경우 해당 정보를 안전한 방법으로 직접 전달해 주시기 바랍니다.  

다른 도움이 필요하시면 언제든 알려 주세요!


**확인**: 이번엔 저장된 메시지에 전화번호·주민등록번호 원문이 없고 `[...마스킹됨]`으로 대체되어 있습니다. 같은 그래프, 같은 체크포인터를 쓰지만, **애초에 마스킹된 텍스트만 그래프에 들어갔기 때문에** 저장소 자체에도 원본 PII가 존재하지 않습니다.

## 정리

| | A (가드레일 없음) | B (가드레일 추가) |
| --- | --- | --- |
| 그래프/체크포인터 코드 | 강의 원본 그대로 | 완전히 동일 (수정 없음) |
| `invoke()`에 들어가는 메시지 | 사용자 입력 원문 | `mask_pii()`를 통과한 마스킹된 텍스트 |
| 체크포인터에 저장된 내용 | 주민등록번호·전화번호 원문 그대로 | `[...마스킹됨]`으로 대체됨 |

**기억할 점**
- 이 방식은 "저장되기 전에" 막는 것이므로, **한 번 원문으로 저장된 뒤에 사후 마스킹하는 것보다 훨씬 안전**합니다(사후 마스킹은 백업본, 로그, 복제본 등 원본이 이미 퍼진 모든 곳까지 따라가서 지울 수 없습니다).
- 실무에서는 정규식 마스킹 외에도 (여유 시간이 있다면) 저장소 자체를 암호화하거나, 사용자의 삭제 요청(예: GDPR "잊혀질 권리")을 처리하는 정책까지 확장할 수 있습니다 — [`2. langgraph/2. Memory`](../2.%20langgraph/2.%20Memory/)의 암호화된 SQLite/PostgreSQL Memory 노트북들을 참고하세요.